In [ ]:
import os
from langchain_core.documents import Document

from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
from dotenv import load_dotenv
import getpass
load_dotenv(".env.RAG")

# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embd = GoogleGenerativeAIEmbeddings(model="models/gemini-2.5-flash")






In [ ]:

from langchain_huggingface import HuggingFaceEmbeddings

embd = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
# file_path = "./Data/Res.pdf"
folder_path="./Data"

loader = DirectoryLoader(
    folder_path,
    glob="**/*",


)


docs = loader.load()
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Md.Ghalib Faruqe

Adresse : Matikata, Dhaka Cantonment, Dhaka Phone : 01719814500 Email : ghalib.arnob@gmail.com LinkedIn : linkedin.com/in/ghalibfaruqe

American International University – Bangladesh

{'source': 'Data\\Res.pdf'}


- used clear_func to formate the text for better embeding

In [ ]:

import re
def cleaner_func(docs):

  if isinstance(docs, list):
    cleared=[]
    for doc in docs:
      if isinstance(doc, str):
         doc = re.sub(r'\n+', '\n', doc)
         doc = re.sub(r'\s+', ' ', doc)
         cleared.append(doc.strip())
      else:
        cleared.append(doc)
    return cleared

    

  elif isnstance(docs, str):
    doc= ''.join(str(d)for d in docs)

  docs = re.sub(r'\n+', '\n', docs)
  docs = re.sub(r'\s+', ' ', docs)
  return docs.strip()

cleaned_docs = cleaner_func(docs)

In [ ]:
text_split = RecursiveCharacterTextSplitter(
    chunk_size= 400, chunk_overlap=100, add_start_index=True

)
splits = text_split.split_documents(cleaned_docs)
print(len(splits))

120


In [ ]:
from langchain_chroma import Chroma
vector_store = Chroma.from_documents(
    documents=splits,
    collection_name="Lang_Rag",
    embedding=embd,
    persist_directory="./Rag/chroma_DB",
    
)


In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":6}
)

docs = retriever.invoke(
    "Which university is mentioned in the resume education section?"
)

for doc in docs:
    print(doc.page_content)
